# Lesson 5: Working with Vector Data

This notebook focuses on **vector** data, exploring vector data operations, exploration, and visualization.

## Learning Objectives:
- Work on conducting exploratory vector data analysis
- Create basic and customized visualizations of vector data
- Conduct some vector operations

Let's get started by loading the necessary packages!



In [1]:
# Install and load necessary packages
library(sf)
library(dplyr)
library(ggplot2)

# Load the Philippines shapefile from local data
regions <- st_read("../../geospatial-data/simplified/regions_simplified.shp")

# Get roads provincial capitals of the Philippines
cities <- st_read("../../geospatial-data/phl_admp_adm2_capitals/phl_admp_adm2_capitals.shp")

# Get philippines roads, limiting them to primary and secondary roads
roads <- st_read("../../geospatial-data/hotosm_phl_roads_lines_shp/hotosm_phl_roads_lines_shp.shp") %>% 
  filter(highway %in% c("primary", "secondary"))




Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.4.0; sf_use_s2() is TRUE




Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Reading layer `regions_simplified' from data source 
  `/home/user/course-4-environmental-analysis/module-12/code-along-notebooks/data/simplified/regions_simplified.shp' 
  using driver `ESRI Shapefile'
Simple feature collection with 17 features and 11 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 114.2779 ymin: 4.587353 xmax: 126.605 ymax: 21.12182
Geodetic CRS:  WGS 84


Reading layer `phl_admp_adm2_capitals' from data source 
  `/home/user/course-4-environmental-analysis/module-12/code-along-notebooks/data/phl_admp_adm2_capitals/phl_admp_adm2_capitals.shp' 
  using driver `ESRI Shapefile'
Simple feature collection with 83 features and 7 fields
Geometry type: POINT
Dimension:     XY
Bounding box:  xmin: 118.733 ymin: 5.029701 xmax: 126.2083 ymax: 20.44813
Geodetic CRS:  WGS 84


Reading layer `hotosm_phl_roads_lines_shp' from data source 
  `/home/user/course-4-environmental-analysis/module-12/code-along-notebooks/data/hotosm_phl_roads_lines_shp/hotosm_phl_roads_lines_shp.shp' 
  using driver `ESRI Shapefile'
Simple feature collection with 45421 features and 14 fields
Geometry type: LINESTRING
Dimension:     XY
Bounding box:  xmin: 117.1971 ymin: 4.642292 xmax: 126.5687 ymax: 20.45039
Geodetic CRS:  WGS 84


Let's take a look at the roads and cities data!

In [0]:
print("CITIES")
print(head(cities))


print("ROADS")
print(head(roads))
print(colnames(roads))

print("REGIONS")
print(head(regions))

Let's visualize a few instances of each of these data types:

In [0]:
# Plot the first region
ggplot() +
  geom_sf(data = head(regions, 1), fill = "lightblue", color = "black") +
  ggtitle("First Region")

# Plot the first road
ggplot() +
  geom_sf(data = head(roads, 1), aes(color = highway)) +
  ggtitle("First Road")

# Plot the first city
ggplot() +
  geom_sf(data = head(cities, 1), color = "red", size = 2) +
  ggtitle("First City")


### From our last lecture--what type of vector data are each of these?

Let's make a plot of the Philippines in the background and our roads data

In [0]:
# Plot the roads data
(roads_map <- ggplot() +
  geom_sf(data = regions, fill = "lightgrey", color = "black") +
  geom_sf(data = roads, aes(color = highway), size = 0.8) +
  theme_minimal() +
  labs(title = "Primary and Secondary Roads in the Philippines",
       caption = "Data source: HOT OSM Philippines"))




### Exercise: See if you can make the same map, but with cities added in blue.

In [0]:
# Add the capital cities data to our map of the Philippines! This map should include roads 
# like above and cities in blue
# YOUR CODE HERE - uncomment the below line to start
# (roads_and_cities_map <- roads_map + 


Filtering vector data can be performed using either metadata or geographic criteria.
- Filtering via Metadata
    - You can use the `filter()` function from the `dplyr` package to select features based on attribute data. For example, you can select regions where ADM1ALT1EN is "Central Visayas".
- Filtering via Geography
    - For spatial filtering based on geography, the `sf` package provides functions like `st_intersection()` to select features that intersect with a given geometry. For example, you can select roads that intersect with the Central Visayas region.
- These methods allow you to refine your spatial data based on specific attributes or spatial relationships.


# Filtering Spatial Data

## Filtering by Attributes with dplyr
We can use dplyr functions like `filter()` to select features based on attribute data. This works the same way as filtering regular data frames. For example, we can filter regions by their name or any other attribute in the data.

## Filtering by Geography with Spatial Joins
For spatial filtering, we can use functions from the sf package like `st_intersection()` to select features based on their spatial relationship with other geometries. This allows us to find features that intersect, contain, or have other spatial relationships with a given area.

Both methods can be combined in a tidyverse workflow to create powerful spatial data queries.



In [0]:
# Filter the Philippines provinces to Central Visayas
central_visayas <- regions %>%
  filter(ADM1ALT1EN == "Central Visayas")

# Filter roads that intersect with Central Visayas
visayas_roads <- st_intersection(roads, central_visayas)

# Filter cities that are within Central Visayas and add coordinates for labeling
visayas_cities <- st_intersection(cities, central_visayas) %>%
  mutate(
    lon = st_coordinates(.)[,1],
    lat = st_coordinates(.)[,2]
  )

# Create a map showing only Central Visayas with its roads and capital, including city names
(visayas_map <- ggplot() +
  geom_sf(data = central_visayas, fill = "lightgrey", color = "black") +
  geom_sf(data = visayas_roads, aes(color = highway), size = 0.8) +
  geom_sf(data = visayas_cities, color = "blue", size = 2) +
  geom_text(
    data = visayas_cities,
    aes(x = lon, y = lat, label = Mun_Name),
    size = 3,
    vjust = -1,
    color = "black"
  ) +
  theme_minimal() +
  labs(
    title = "Roads and Provincial Capital in Central Visayas Region",
    caption = "Data sources: HOT OSM Philippines, PSA-NAMRIA"
  ))


# Other ways to spatially join:

This can also be done using the `st_filter()` function, and we can join spatial data using `st_join()`. For example, we can filter the roads data to only include roads that intersect with the Central Visayas region:
`st_filter(roads, central_visayas)`

If we wanted to add region attributes to the capital cities, we could use the `st_join()` function:
`st_join(cities, regions)`

In [0]:
# You do the same for the national capital region--called "NCR"--plot the roads and cities for that region.
# YOUR CODE HERE 




There aren't any captial cities in the NCR region so let's try a spatial join with the cities data in whole country. Here we will use the `st_join()` function to join the cities data to the regions data and explore the results.

In [0]:
# Then add the region attributes to the cities
cities_with_region <- st_join(cities, regions)

print(colnames(cities_with_region))

 # Zooming in even more
 Now let's zoom in even futher, focusing on the City of San Fernando. What if we wanted to get the roads within 50km of the city? 

 We can do this by using the `st_buffer()` function to create a buffer around the city and then using the `st_filter()` function to filter the roads data to only include roads that intersect with the buffer.

In [0]:
# First, find Cebu City in the visayas_cities dataset
cebu_city <- visayas_cities %>%
  filter(Mun_Name == "CEBU CITY (Capital)")
print(cebu_city)
print(x = table(regions$ADM1ALT1EN))
# Create a 50km buffer around Cebu City
cebu_city_buffer <- st_buffer(cebu_city, dist = 10000)  # 50,000 meters = 50km

# Filter roads that intersect with the buffer
roads_near_cebu_city <- st_filter(roads, cebu_city_buffer)

# Plot the results
ggplot() +
  geom_sf(data = cebu_city_buffer, fill = "lightblue", alpha = 0.3) +
  geom_sf(data = roads_near_cebu_city, aes(color = highway), size = 0.8) +
  geom_sf(data = cebu_city, color = "red", size = 3) +
  theme_minimal() +
  labs(title = "Roads within 50km of Cebu City",
       subtitle = "Central Visayas Region, Philippines",
       caption = "Data sources: HOT OSM Philippines, PSA-NAMRIA")


# Conclusion
In this lesson, we've explored spatial data manipulation and visualization using R's sf package and ggplot2. We've learned how to:

1. Load and manipulate spatial data with sf
2. Join spatial datasets based on their geometry
3. Filter spatial data based on attributes
4. Create buffers around points of interest
5. Visualize spatial data with different aesthetics using ggplot2

These techniques are fundamental for spatial analysis and can be applied to a wide range of environmental and geographic data science problems, from urban planning to environmental monitoring. The combination of R's tidyverse approach with spatial capabilities makes it a powerful tool for geographic data science.

In the next lesson we'll explore interactions between raster and vector data.
